# Walkthrough: Securing AI Agents with Information-Flow Control

Paper: Costa et al., arXiv:2505.23643v2 (**Fides**).

This notebook maps paper sections to code and runs CPU-only sanity checks. No API key required.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.utils import (
    IntegrityLabel, ConfidentialityLabel, ProductLabel,
    readers_label, security_bottom, security_top,
)
print("imports ok", ROOT)

## Information-flow labels (§4.1)

> "we require that labels L form a lattice with a partial order ⊑ and join operation ⊔"
>
> Confidentiality: L ⊑ H. Integrity: T ⊑ U. Product: ⊤ = (U, H), ⊥ = (T, L).
>
> Readers lattice: join is **set intersection** — `{A,B,C} ⊔ {B,C,D} = {B,C}`.

In [ ]:
T, U = IntegrityLabel.trusted(), IntegrityLabel.untrusted()
assert T.leq(U) and not U.leq(T)
assert T.join(U) == U

L, H = ConfidentialityLabel.low(), ConfidentialityLabel.high()
bot = ProductLabel(T, L)
top = ProductLabel(U, H)
assert bot.leq(top) and bot.join(top) == top

univ = frozenset({"A", "B", "C", "D"})
abc = readers_label(frozenset({"A", "B", "C"}), univ)
bcd = readers_label(frozenset({"B", "C", "D"}), univ)
joined = abc.join(bcd)
assert joined.inner.subset == frozenset({"B", "C"}), joined
print("§4.1 lattice sanity passed", joined)

## Threat model (§2.1)

> Configuration (system message, tool descriptions, LLMs) is trusted.
> The adversary tampers with **tool results** (web pages, email) and may observe egress.
>
> Indirect prompt injection is in-scope. The LLM is **not** a trusted computing base.

## Policies P-T and P-F (§4.3)

> **P-T:** tool call only if generated from trusted inputs. π_f = (T, ⊤).
>
> **P-F:** egress only if every recipient may read the data. π_d = (⊤, R).
>
> Algorithm 5: `if ¬policy(action) then abort` **before** executing the tool.

In [ ]:
from src.evaluate import _load_config, run_undefended, run_with_policy
from src.policy import PT_ONLY, DEMO_POLICIES

cfg = _load_config()
undefended = run_undefended(cfg)
pt = run_with_policy(cfg, PT_ONLY, "P-T")
fides = run_with_policy(cfg, DEMO_POLICIES, "fides", fides=True)

print("undefended ASR", undefended.attack_success, "emails", undefended.emails_to)
print("P-T        ASR", pt.attack_success, "denied", pt.denied, pt.deny_reason)
print("Fides      ASR", fides.attack_success, "TCR", fides.task_complete, "emails", fides.emails_to)

assert undefended.attack_success
assert pt.denied and not pt.attack_success
assert fides.task_complete and not fides.attack_success
print("§1 demo sanity passed")

## Fides HIDE (§5.1 / Algorithm 7)

> Hide a tool-result node when its label is **not ⊑** the current context label.
> Query then uses the **unchanged** context label ℓσ, so later P-T tool calls remain possible.
>
> Appendix D.1: evaluation hides on **integrity** only (`hide_on: integrity` in `configs/base.yaml`).